# SmolLM2 — automated, safe hyperparameter search (Colab)

This notebook uses the existing **Phase 1 `main.py`** training/evaluation pipeline; it does **not** replace the fine-tuner or judge. It prepares many **separately named** 360M / 1.7B experiments and executes them **sequentially**, one GPU job at a time.

**Search:** learning rate, number of epochs and LoRA rank, with identical 1,000-ticket teacher-labeled data, seed, split logic and evaluation settings. Two presets: **quick** (4 experiments: original and lower LR for each model), **extended** (8 experiments: also fewer epochs and rank 32). Edit `CUSTOM_TRIALS` to try other combinations. **Never reuses or overwrites** your original `smol360_n1000` or `smol17b_n1000` results.

**Workflow:**

1. Run setup cells through the compatibility patch.
2. Set `SEARCH_PRESET`, `DRY_RUN`, and `MAX_NEW_RUNS_PER_EXECUTION` in **Search controls**.
3. Run **Load shared dataset**, **Build plan**, and **Execute planned trials**.
4. Run **Compare validation results** to see the best *observed validation* score and speed.
5. In a new Colab session, repeat setup and run the plan/execution cells: **completed experiments are skipped**. Incomplete experiment outputs are **never overwritten automatically**.

**Important:** The first run defaults to `DRY_RUN=True`, so it will **not** train, judge, or spend external API credits until it's changed to `False`. Multiple model evaluations may make paid Anthropic calls. GPU training may take substantial time and Drive storage. Colab runtime may disconnect between trials; running again advances to the next pending experiment.

**Selection integrity:** Rankings require an explicit **validation** row in `judge_summary.csv`. No silent fallback to test scores. Your upstream pipeline may still *produce* test evaluations for every trial. Avoid using those scores to tune parameters. The winner is provisional until you examine held-out test results. Judge variance and only ~37 clusters mean small differences are not strong evidence.

**Compatibility:** Built from the supplied `SmolLM2_Only_Colab_Rerun_Fix.ipynb` and its config/CSV contracts. Exact behavior of your GitHub branch's training/evaluation internals could not be executed in this environment. If `eval_all_splits=True` doesn't write validation rows in your checkout, the comparison will deliberately flag that as a blocker rather than invent rankings.

## 1. Environment setup — run these cells once per Colab session

In [ ]:
# ── CELL 1: Verify GPU ────────────────────────────────────────────────────────
# Expected: Tesla T4, ~15 GB VRAM
# If you see K80 or < 14 GB: Runtime → Disconnect and delete runtime → reconnect
!nvidia-smi
import torch
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    icon = '✅' if vram >= 14 else '⚠️ '
    print(f'\n{icon} GPU: {name}  ({vram:.0f} GB VRAM)')
    if vram < 14:
        print('   You have a K80 (12 GB). Reconnect to get a T4 (16 GB).')
else:
    print('\n❌ No GPU — go to Runtime → Change runtime type → T4 GPU')

Sat Nov  7 03:56:38 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   52C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# ── CELL 2: Mount Google Drive ────────────────────────────────────────────────
# A popup asks for permissions — click Allow on everything (normal Google behaviour).
from google.colab import drive
import os
drive.mount('/content/drive')
if os.path.exists('/content/drive/MyDrive'):
    print('✅ Drive mounted')
else:
    print('❌ Mount failed — run this cell again')

Mounted at /content/drive
✅ Drive mounted


In [ ]:
# ── Clone or update your project's checkout (without putting the PAT in Git URL) ──
# The existing repo was confirmed to be public at the time this notebook was built.
# If your branch becomes private, authenticate with GitHub separately before cloning.
from pathlib import Path
import os, subprocess, sys

BRANCH_NAME = 'btt-setup-js-test'  # Must match the branch used by your original notebook.
REPO_ORG = 'Break-Through-Tech'
REPO_NAME = 'Automation-Anywhere-1B-domain-specific-theme-labeling-via-slm-distillation'
REPO_DIR = Path('/content/project')
REPO_URL = f'https://github.com/{REPO_ORG}/{REPO_NAME}.git'

def git_run(args, cwd=None):
    result = subprocess.run(args, cwd=cwd, text=True, capture_output=True)
    if result.returncode:
        raise RuntimeError(f'Git command failed: {args[:3]}\n{result.stderr[-2000:]}')
    return result.stdout.strip()

if not (REPO_DIR / '.git').exists():
    if REPO_DIR.exists():
        raise RuntimeError('Existing /content/project is not a Git checkout. Inspect it before removing it.')
    git_run(['git', 'clone', '--branch', BRANCH_NAME, REPO_URL, str(REPO_DIR)])
else:
    # Repair remote URL if an earlier notebook stored credentials in origin URL.
    git_run(['git', 'remote', 'set-url', 'origin', REPO_URL], cwd=REPO_DIR)
    git_run(['git', 'fetch', 'origin', BRANCH_NAME], cwd=REPO_DIR)
    git_run(['git', 'checkout', BRANCH_NAME], cwd=REPO_DIR)
    git_run(['git', 'pull', '--ff-only', 'origin', BRANCH_NAME], cwd=REPO_DIR)

if (REPO_DIR / 'code' / 'main.py').exists():
    CODE_DIR = REPO_DIR / 'code'
elif (REPO_DIR / 'main.py').exists():
    CODE_DIR = REPO_DIR
else:
    raise FileNotFoundError(f'main.py not found in {REPO_DIR} or code/')

os.environ['SLM_CODE_DIR'] = str(CODE_DIR)
os.chdir(CODE_DIR)
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))
print('Branch:', BRANCH_NAME)
print('Git revision:', git_run(['git','rev-parse','HEAD'], cwd=REPO_DIR))
print('Code directory:', CODE_DIR)

Branch: btt-setup-js-test
Git revision: 09629936c58a17b92807ad422ef1c755132c3196
Code directory: /content/project/code


In [ ]:
# ── CELL 4: Install dependencies ──────────────────────────────────────────────
# Reads requirements files from the code directory found in Cell 3.
# Takes 3–4 minutes. Normal to see some warnings.
import os, subprocess, sys

CODE_DIR = os.environ.get('SLM_CODE_DIR', '/content/project/code')
print(f'Installing from: {CODE_DIR}')

def pip_install(filename):
    path = f'{CODE_DIR}/{filename}'
    if not os.path.exists(path):
        print(f'❌ {filename} not found at {path}')
        print('   Make sure Cell 3 ran successfully first.')
        return False
    print(f'\nInstalling {filename} ...')
    result = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', '-r', path],
        capture_output=True, text=True
    )
    tail = (result.stdout + result.stderr).strip().split('\n')
    for line in tail[-5:]:
        if line.strip():
            print(f'  {line}')
    if result.returncode != 0:
        print(f'❌ pip failed for {filename}')
        return False
    print(f'✅ {filename} done')
    return True

ok1 = pip_install('requirements.txt')
ok2 = pip_install('requirements_colab.txt')
print('\n✅ All dependencies installed' if (ok1 and ok2) else '\n⚠️  Check errors above')

Installing from: /content/project/code

Installing requirements.txt ...
✅ requirements.txt done

Installing requirements_colab.txt ...
✅ requirements_colab.txt done

✅ All dependencies installed


In [ ]:
# ── CELL 5: Drive folder structure and HuggingFace cache ─────────────────────
import os

DRIVE_ROOT = '/content/drive/MyDrive/slm-distillation'
for d in [f'{DRIVE_ROOT}/data/raw', f'{DRIVE_ROOT}/data/processed',
          f'{DRIVE_ROOT}/data/checkpoints', f'{DRIVE_ROOT}/outputs',
          f'{DRIVE_ROOT}/hf_cache']:
    os.makedirs(d, exist_ok=True)

os.environ['HF_HOME'] = f'{DRIVE_ROOT}/hf_cache'
os.environ['DRIVE_ROOT'] = DRIVE_ROOT

print(f'✅ Drive folders ready under {DRIVE_ROOT}')
print(f'✅ HF model cache → {os.environ["HF_HOME"]}')

✅ Drive folders ready under /content/drive/MyDrive/slm-distillation
✅ HF model cache → /content/drive/MyDrive/slm-distillation/hf_cache


In [ ]:
# ── CELL 6: Load API keys from Colab Secrets ──────────────────────────────────
import os
from google.colab import userdata

def load_secret(name, required=True):
    try:
        val = userdata.get(name)
        if val:
            os.environ[name] = val
            print(f'  ✅ {name}')
            return True
        print(f'  ⚠️  {name} is empty — {"add value in 🔑 Secrets" if required else "optional"}')
    except Exception:
        print(f'  ❌ {name} not found — {"open 🔑 Secrets and toggle Notebook access ON" if required else "optional"}')
    return not required

print('Loading secrets:')
all_ok = all([
    load_secret('ANTHROPIC_API_KEY', required=True),
    load_secret('HF_TOKEN',          required=True),
    load_secret('OPENAI_API_KEY',    required=False),
])
print('\n✅ Required secrets loaded' if all_ok else '\n❌ Fix missing secrets before running the pipeline')

Loading secrets:
  ✅ ANTHROPIC_API_KEY
  ✅ HF_TOKEN
  ❌ OPENAI_API_KEY not found — optional

✅ Required secrets loaded


In [ ]:
# ── CELL 7: Verify everything is ready ───────────────────────────────────────
import os, torch

code_dir  = os.environ.get('SLM_CODE_DIR', '')
drive_ok  = os.path.exists('/content/drive/MyDrive')
gpu_ok    = torch.cuda.is_available()
vram_ok   = gpu_ok and torch.cuda.get_device_properties(0).total_memory > 14e9

checks = [
    ('T4 GPU (≥14 GB VRAM)',   vram_ok),
    ('Drive mounted',          drive_ok),
    ('Code directory found',   bool(code_dir) and os.path.exists(code_dir)),
    ('main.py present',        os.path.exists(f'{code_dir}/main.py') if code_dir else False),
    ('ANTHROPIC_API_KEY set',  'ANTHROPIC_API_KEY' in os.environ),
    ('HF_TOKEN set',           'HF_TOKEN' in os.environ),
    ('HF cache on Drive',      os.environ.get('HF_HOME','').startswith('/content/drive')),
]

print('Setup verification:')
print(f'  Code directory: {code_dir or "NOT SET"}')
print()
all_ok = True
for label, ok in checks:
    print(f'  {"✅" if ok else "❌"} {label}')
    if not ok:
        all_ok = False

print()
print('🚀 Ready! Scroll down to run the pipeline.' if all_ok else
      '⚠️  Fix ❌ items before running the pipeline.')

Setup verification:
  Code directory: /content/project/code

  ✅ T4 GPU (≥14 GB VRAM)
  ✅ Drive mounted
  ✅ Code directory found
  ✅ main.py present
  ✅ ANTHROPIC_API_KEY set
  ✅ HF_TOKEN set
  ✅ HF cache on Drive

🚀 Ready! Scroll down to run the pipeline.


---
### Compatibility patch (run after setup)

The uploaded repository's model loader can attach a LoRA adapter to the base model, which would invalidate a clean baseline. The tokenizer stage can also load more weights than necessary. This cell applies guarded fixes before **SmolLM2** experiments. It never edits the GitHub remote.

If a patch fails, stop and inspect the source. Do not continue with unverified baseline/adapter behavior.


In [ ]:
from pathlib import Path
import os

CODE = Path(os.environ.get('SLM_CODE_DIR', '/content/project/code'))
assert (CODE / 'main.py').exists(), 'Run setup Cell 3 first.'
trainer = CODE / 'phase1/finetuning/trainer.py'
pipeline = CODE / 'phase1/pipeline.py'

# Create backups of the CURRENT checkout before editing it.
for p in (trainer, pipeline):
    backup = p.with_suffix('.py.bak_smol_setup')
    if not backup.exists():
        backup.write_text(p.read_text())

def apply_patch(path, original, replacement, label):
    contents = path.read_text()
    if replacement in contents:
        print('Already applied:', label)
        return
    if original not in contents:
        raise RuntimeError(f'Patch location not found: {label} in {path}')
    path.write_text(contents.replace(original, replacement, 1))
    print('Applied:', label)

apply_patch(trainer,
    '        model = FastLanguageModel.get_peft_model(\n            model,\n            r=lora_cfg["r"],\n            lora_alpha=lora_cfg["lora_alpha"],\n            target_modules=target_modules,\n            lora_dropout=lora_cfg["lora_dropout"],\n            bias=lora_cfg["bias"],\n            use_gradient_checkpointing="unsloth",\n        )\n',
    '        # Keep model unadapted here: also used for baseline inference.\n',
    'Load clean base model during baseline inference')

apply_patch(trainer,
    '    from peft import get_peft_model, LoraConfig, TaskType',
    '    from peft import get_peft_model, LoraConfig, TaskType, PeftModel',
    'Import PeftModel to detect a real attached adapter')

apply_patch(trainer,
    '    if not _is_unsloth_model(model):',
    '    if not isinstance(model, PeftModel):',
    'Apply LoRA during training, not during base-model load')

apply_patch(trainer,
    '        bnb_4bit_compute_dtype=torch.bfloat16,',
    '        bnb_4bit_compute_dtype=(torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16),',
    'Select appropriate 4-bit compute dtype')

apply_patch(trainer,
    '        model_id, quantization_config=bnb_config, device_map="auto"',
    '        model_id, quantization_config=bnb_config, device_map="auto", attn_implementation="eager"',
    'Fallback eager attention on older GPUs')

apply_patch(pipeline,
    '        _, tokenizer_tmp = load_model_and_tokenizer(cfg)',
    '        from transformers import AutoTokenizer\n        tokenizer_tmp = AutoTokenizer.from_pretrained(cfg["student_slm"]["model_id"].strip())',
    'Avoid extra full model load for tokenizer')

apply_patch(pipeline,
    "datetime.now().strftime('%Y%m%d_%H%M')",
    "datetime.now().strftime('%Y%m%d_%H%M%S')",
    'Unique run ID down to seconds')

import ast
for p in (trainer, pipeline):
    ast.parse(p.read_text(), filename=str(p))
print('\nSUCCESS: patches applied and Python syntax verified.')


Applied: Load clean base model during baseline inference
Applied: Import PeftModel to detect a real attached adapter
Applied: Apply LoRA during training, not during base-model load
Applied: Select appropriate 4-bit compute dtype
Applied: Fallback eager attention on older GPUs
Applied: Avoid extra full model load for tokenizer
Applied: Unique run ID down to seconds

SUCCESS: patches applied and Python syntax verified.


## 2. Search controls — edit only this cell for ordinary experiments

- **Quick:** 4 trials, 2 per model; compares learning rate 2e-4 vs 1e-4 while holding all else fixed.
- **Extended:** 8 trials; also tests 2 epochs (vs 3) and LoRA rank 32 with alpha 32 (keeps alpha/r constant) for each model.
- **Custom:** change the `CUSTOM_TRIALS` list to explicitly set combinations. Every unique model/parameter/config/dataset/code combination gets a stable experiment ID.

`MAX_NEW_RUNS_PER_EXECUTION=2` ensures a single click does not launch the whole search unexpectedly. Set to 8 or more to run all pending trials sequentially. `DRY_RUN=True` runs *zero* training trials.

In [ ]:
# ── All ordinary user controls ────────────────────────────────────────────────
SEARCH_PRESET = "quick"              # 'quick', 'extended', or 'custom'
DRY_RUN = False                      # MUST set False to permit training and judge API calls
MAX_NEW_RUNS_PER_EXECUTION = 3      # upper limit on NEW training trials per execution
INCOMPLETE_ACTION = 'skip'          # 'skip' (safe default), or 'archive_and_restart'
STOP_ON_ERROR = True

# Fixed shared conditions for fair comparison (do not change mid-search).
N_SAMPLES = 1000
SPLIT_SEED = 42
MAX_SEQ_LENGTH = 2048
BATCH_SIZE = 1
GRAD_ACCUM = 16

MODEL_IDS = {
    '360m': 'HuggingFaceTB/SmolLM2-360M-Instruct',
    '17b': 'HuggingFaceTB/SmolLM2-1.7B-Instruct',
}

# Only used if SEARCH_PRESET == 'custom'.
# Valid trial keys: model, label, learning_rate, epochs, lora_r, lora_alpha,
# and optional lora_dropout. Keep LoRA rank/alpha reasonable for your GPU.
CUSTOM_TRIALS = [
    {'model':'360m', 'label':'custom_360_lr1e4', 'learning_rate':1e-4, 'epochs':3, 'lora_r':16, 'lora_alpha':16},
    {'model':'17b',  'label':'custom_17b_lr1e4', 'learning_rate':1e-4, 'epochs':3, 'lora_r':16, 'lora_alpha':16},
]

assert SEARCH_PRESET in {'quick', 'extended', 'custom'}
assert INCOMPLETE_ACTION in {'skip', 'archive_and_restart'}
assert isinstance(DRY_RUN, bool)
assert isinstance(MAX_NEW_RUNS_PER_EXECUTION, int) and MAX_NEW_RUNS_PER_EXECUTION >= 0
print('Preset:', SEARCH_PRESET, '| DRY_RUN:', DRY_RUN,
      '| max new runs this execution:', MAX_NEW_RUNS_PER_EXECUTION)

Preset: quick | DRY_RUN: False | max new runs this execution: 3


## 3. Locate and fingerprint the teacher-labeled dataset

Uses the same three saved CSV files as your original notebook, **without generating labels again**. If the shared snapshot is missing and multiple candidate datasets exist, provide an explicit path; do not silently choose a different source. The shared snapshot and code/config fingerprints become part of each experiment ID.

In [ ]:
from pathlib import Path
import os, sys, subprocess, shutil, hashlib, json, yaml, pandas as pd
from datetime import datetime, timezone

CODE = Path(os.environ.get('SLM_CODE_DIR', '/content/project/code'))
REPO = Path('/content/project')
DRIVE_ROOT = Path('/content/drive/MyDrive/slm-distillation')
EXPERIMENTS = DRIVE_ROOT / 'experiments'
SEARCH_ROOT = EXPERIMENTS / 'hpo_smol_search'
SHARED_DATA = DRIVE_ROOT / 'shared_datasets/bitext_n1000_teacher/data/processed'
REQUIRED_FILES = ('bitext_clustered.csv', 'bitext_grouped.csv', 'bitext_labeled.csv')
# Optional explicit source folder containing the REQUIRED_FILES, if shared data missing.
DATASET_SOURCE = None

assert (CODE / 'main.py').is_file(), 'Run environment setup and clone first.'
assert (CODE / 'configs/phase1_config.yaml').is_file(), 'Missing base config in this Git checkout.'
assert (DRIVE_ROOT / 'data').exists(), 'Mount Drive / run setup first.'

def all_required(folder):
    return all((folder / name).is_file() for name in REQUIRED_FILES)

if not all_required(SHARED_DATA):
    if DATASET_SOURCE:
        candidate = Path(DATASET_SOURCE)
        assert all_required(candidate), f'Explicit source lacks required CSVs: {candidate}'
    else:
        candidates = []
        for labeled in EXPERIMENTS.glob('*/data/processed/bitext_labeled.csv'):
            folder = labeled.parent
            if not all_required(folder):
                continue
            try:
                groups = pd.read_csv(folder / 'bitext_grouped.csv')
                if len(groups) == 37: # original 1,000-ticket snapshot
                    candidates.append(folder)
            except Exception:
                continue
        if len(candidates) != 1:
            raise RuntimeError(
                f'Shared dataset missing at {SHARED_DATA}. Found {len(candidates)} '
                'plausible candidates. Set DATASET_SOURCE to your trusted folder.\n' +
                '\n'.join(str(c) for c in candidates[:20]))
        candidate = candidates[0]
    SHARED_DATA.mkdir(parents=True, exist_ok=True)
    for p in sorted(candidate.iterdir()):
        if p.is_file() and p.suffix.lower() in {'.csv', '.json', '.jsonl'}:
            shutil.copy2(p, SHARED_DATA / p.name)
    print('Created shared snapshot from:', candidate)

assert all_required(SHARED_DATA), f'Missing input file in {SHARED_DATA}'

def sha_file(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1024*1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

dataset_hashes = {p.name: sha_file(p) for p in sorted(SHARED_DATA.iterdir())
                  if p.is_file() and p.suffix.lower() in {'.csv', '.json', '.jsonl'}}
assert set(REQUIRED_FILES).issubset(dataset_hashes)
assert N_SAMPLES == 1000, 'This shared dataset is the original n1000 snapshot; use a separate snapshot for other sample counts.'
DATASET_SHA = hashlib.sha256(json.dumps(dataset_hashes,sort_keys=True).encode()).hexdigest()
BASE_YAML = CODE / 'configs/phase1_config.yaml'
BASE_CFG_SHA = sha_file(BASE_YAML)
BASE_CFG = yaml.safe_load(BASE_YAML.read_text())
GIT_SHA = subprocess.run(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True,
                         capture_output=True, check=True).stdout.strip()
PATCHED_FILE_HASHES = {
    name: sha_file(CODE / name)
    for name in ['phase1/finetuning/trainer.py','phase1/pipeline.py']
}
print('Shared dataset:', SHARED_DATA)
cluster_count = len(pd.read_csv(SHARED_DATA / 'bitext_grouped.csv'))
assert cluster_count == 37, f'Expected 37 clusters in saved n1000 dataset, got {cluster_count}'
print('Cluster count:', cluster_count)
print('Dataset SHA256:', DATASET_SHA[:16])
print('Base config SHA256:', BASE_CFG_SHA[:16])
print('Git revision:', GIT_SHA[:12])

Shared dataset: /content/drive/MyDrive/slm-distillation/shared_datasets/bitext_n1000_teacher/data/processed
Cluster count: 37
Dataset SHA256: 5737ceebbe01fb37
Base config SHA256: e30282986bdd01dd
Git revision: 09629936c58a


## 4. Generate a reproducible trial plan

Every experiment name includes a digest of the model, training settings, source dataset, base configuration, and the patched training code. When any of those change, you get a **new** directory. Existing experiments and their saved configs are never silently replaced.

In [ ]:
import copy, re, json, hashlib, pandas as pd

VARS = [
    {'label': 'base',       'learning_rate':2e-4, 'epochs':3, 'lora_r':16, 'lora_alpha':16},
    {'label': 'lr_1e4',     'learning_rate':1e-4, 'epochs':3, 'lora_r':16, 'lora_alpha':16},
    {'label': 'epochs_2',   'learning_rate':2e-4, 'epochs':2, 'lora_r':16, 'lora_alpha':16},
    {'label': 'rank_32',    'learning_rate':2e-4, 'epochs':3, 'lora_r':32, 'lora_alpha':32},
]

if SEARCH_PRESET == 'quick':
    RAW_TRIALS = [{'model':model, **variant} for model in MODEL_IDS for variant in VARS[:2]]
elif SEARCH_PRESET == 'extended':
    RAW_TRIALS = [{'model':model, **variant} for model in MODEL_IDS for variant in VARS]
else:
    RAW_TRIALS = list(CUSTOM_TRIALS)
assert RAW_TRIALS, 'No trials configured for this search.'

KNOWN_TRIAL_KEYS = {'model','label','learning_rate','epochs','lora_r','lora_alpha','lora_dropout'}

def validate_trial(t):
    unknown = set(t) - KNOWN_TRIAL_KEYS
    assert not unknown, f'Unknown trial keys {unknown}: {t}'
    assert t['model'] in MODEL_IDS, f'Unknown model {t["model"]}'
    assert t['learning_rate'] > 0 and t['learning_rate'] <= 0.001
    assert isinstance(t['epochs'], int) and t['epochs'] >= 1
    assert isinstance(t['lora_r'], int) and t['lora_r'] in (4,8,16,32,64)
    assert isinstance(t['lora_alpha'], int) and t['lora_alpha'] > 0
    assert 0 <= t.get('lora_dropout',0) < 1


def canonical(obj): return json.dumps(obj, sort_keys=True, separators=(',', ':'), allow_nan=False)

TRIAL_PLAN = []
seen = set()
for trial in RAW_TRIALS:
    validate_trial(trial)
    t = dict(trial)
    t.setdefault('lora_dropout', BASE_CFG['lora'].get('lora_dropout', 0))
    # Disallow collisions where labels differ but actual params are identical.
    signature = {
        'model_id': MODEL_IDS[t['model']],
        'params': {k:t[k] for k in ('learning_rate','epochs','lora_r','lora_alpha','lora_dropout')},
        'shared': {'n_samples':N_SAMPLES,'seed':SPLIT_SEED,'max_seq_length':MAX_SEQ_LENGTH,
                   'batch_size':BATCH_SIZE,'grad_accum':GRAD_ACCUM},
        'source': {'dataset_sha':DATASET_SHA, 'base_cfg_sha':BASE_CFG_SHA,
                   'git_sha':GIT_SHA,'patches':PATCHED_FILE_HASHES},
    }
    digest = hashlib.sha256(canonical(signature).encode()).hexdigest()[:12]
    if digest in seen:
        raise ValueError('Duplicate model/hyperparameter combination: ' + repr(t))
    seen.add(digest)
    safe_label = re.sub('[^a-z0-9_]+','_',t['label'].lower())[:32].strip('_')
    exp_name = f'hpo_{t["model"]}_{safe_label}_{digest}'
    TRIAL_PLAN.append({'trial':t,'signature':signature,'experiment':exp_name,
                       'folder':EXPERIMENTS / exp_name})

plan_table = pd.DataFrame([
    {'experiment':r['experiment'],'model':r['trial']['model'],
     'lr':r['trial']['learning_rate'], 'epochs':r['trial']['epochs'],
     'LoRA r':r['trial']['lora_r'], 'LoRA alpha':r['trial']['lora_alpha'],
     'existing outputs':(r['folder'] / 'outputs').exists()}
    for r in TRIAL_PLAN
])
print(f'Planned trials: {len(TRIAL_PLAN)} | new trials allowed this execution: {MAX_NEW_RUNS_PER_EXECUTION}')
display(plan_table)

Planned trials: 4 | new trials allowed this execution: 3


,experiment,model,lr,epochs,LoRA r,LoRA alpha,existing outputs
0,hpo_360m_base_a54f178d4227,360m,0.0002,3,16,16,True
1,hpo_360m_lr_1e4_57c233a7b33d,360m,0.0001,3,16,16,False
2,hpo_17b_base_c66db95a7295,17b,0.0002,3,16,16,False
3,hpo_17b_lr_1e4_e503eba19d34,17b,0.0001,3,16,16,False


## 5. Execute sequentially; skip completed experiments; preserve incomplete runs

- **Dry run:** prints planned trials, no model loads or API calls.
- **Real run:** stages config + saved teacher labels for each trial and invokes the same `main.py --phase 1 --device_mode colab` as your original notebook.
- **On rerun:** complete outputs are skipped; incomplete outputs are skipped by default and flagged for inspection. If you intentionally change `INCOMPLETE_ACTION` to `archive_and_restart`, ONLY the incomplete trial's `outputs` is timestamp-archived (not deleted) before restarting.
- Logs and a `run_status.json` are saved in each trial folder. External evaluation and training time depend on the existing project code.

**Run this cell more than once** to advance through the queue. Keep `DRY_RUN=False` if you want actual training; adjust the maximum trials per click depending on GPU time and API budget.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import copy, json, os, shutil, subprocess, sys, time, yaml, pandas as pd

EXPECTED_CSVS = ('judge_summary.csv','metrics_summary.csv','business_eval.csv')
VAL_ALIASES = {'val','valid','validation','dev'}

def now_utc():
    return datetime.now(timezone.utc).isoformat()

def complete_eval_dirs(outputs):
    if not outputs.is_dir(): return []
    found=[]
    for run in sorted(outputs.iterdir()):
        ev=run/'evaluation'
        if run.is_dir() and all((ev/name).is_file() and (ev/name).stat().st_size > 0
                                    for name in EXPECTED_CSVS):
            found.append(ev)
    return found

def eval_has_validation(ev):
    try:
        j = pd.read_csv(ev / 'judge_summary.csv')
        ok = ('split' in j and 'model' in j and 'composite' in j and
              ((j['split'].astype(str).str.lower().str.strip().isin(VAL_ALIASES)) &
               (j['model'].astype(str).str.lower().str.strip() == 'finetuned')).any())
        return bool(ok)
    except Exception:
        return False

def outputs_state(dest):
    outs = dest/'outputs'
    if not outs.exists(): return 'NEW', None
    if not outs.is_dir(): return 'INCOMPLETE', None
    valid = complete_eval_dirs(outs)
    if valid:
        # 'Complete' here = comparison CSVs exist; validation may still be missing.
        newest = max(valid, key=lambda ev: ev.stat().st_mtime)
        return ('COMPLETE' if eval_has_validation(newest) else 'COMPLETE_NO_VALIDATION'), newest
    return 'INCOMPLETE', None

def build_config(rec):
    trial=rec['trial']
    cfg=copy.deepcopy(BASE_CFG)
    for sect in ('training','lora','qlora','student_slm','dataset','colab','paths','pipeline','evaluation'):
        assert isinstance(cfg.get(sect), dict), f'Base config missing section: {sect}'
    cfg['device_mode']='colab'
    cfg['student_slm']['model_id']=MODEL_IDS[trial['model']]
    cfg['student_slm']['max_seq_length']=MAX_SEQ_LENGTH
    cfg['dataset']['n_samples']=N_SAMPLES
    cfg['seed']=SPLIT_SEED
    cfg['training'].update({
        'num_train_epochs':trial['epochs'],
        'learning_rate':float(trial['learning_rate']),
        'per_device_train_batch_size':BATCH_SIZE,
        'gradient_accumulation_steps':GRAD_ACCUM,
        'fp16':True,'bf16':False,
    })
    cfg['lora'].update({'r':trial['lora_r'], 'lora_alpha':trial['lora_alpha'],
                        'lora_dropout':trial['lora_dropout']})
    cfg['qlora']['load_in_4bit']=True
    cfg['colab']['drive_root_override']=str(rec['folder'])
    cfg['colab']['cache_model_on_drive']=False
    cfg['paths']['hf_cache']='/content/hf_cache_smol_models'
    cfg['pipeline'].update({
        'run_clustering':False, 'run_preprocessing':False, 'run_label_generation':False,
        'run_finetuning':True, 'run_baseline_eval':True, 'run_finetuned_eval':True,
        'run_llm_judge':True, 'run_business_eval':True,
    })
    cfg['evaluation'].update({
        'existing_run_dir':None, 'eval_all_splits':True, 'run_bertscore':False,
    })
    return cfg

def stage_trial(rec):
    dest=rec['folder']
    dest.mkdir(parents=True,exist_ok=True)
    payload=yaml.safe_dump(build_config(rec),sort_keys=False)
    saved=dest/'experiment_config.yaml'
    local_cfg=CODE/'configs'/f'{rec["experiment"]}.yaml'
    manifest_file=dest/'manifest.json'
    expected_manifest={'name':rec['experiment'], 'trial':rec['trial'],
                       'signature':rec['signature'], 'config_sha256':hashlib.sha256(payload.encode()).hexdigest(),
                       'source_dataset':str(SHARED_DATA)}
    if saved.exists() and saved.read_text()!=payload:
        raise RuntimeError(f'Config drift detected at {saved}. Never overwrite an existing trial.')
    if manifest_file.exists() and json.loads(manifest_file.read_text()) != expected_manifest:
        raise RuntimeError(f'Manifest mismatch at {manifest_file}. Inspect before continuing.')
    if local_cfg.exists() and local_cfg.read_text()!=payload:
        raise RuntimeError(f'Local config collision at {local_cfg}.')
    if not saved.exists(): saved.write_text(payload)
    if not manifest_file.exists(): manifest_file.write_text(json.dumps(expected_manifest,indent=2))
    local_cfg.write_text(payload)
    processed=dest/'data/processed'
    processed.mkdir(parents=True,exist_ok=True)
    for src in sorted(SHARED_DATA.iterdir()):
        if not src.is_file() or src.suffix.lower() not in {'.csv','.json','.jsonl'}:
            continue
        target=processed/src.name
        if target.exists():
            if target.stat().st_size!=src.stat().st_size or sha_file(target)!=sha_file(src):
                raise RuntimeError(f'Copied dataset differs from shared snapshot: {target}')
        else:
            shutil.copy2(src,target)
    return local_cfg

def write_status(rec, status, **extra):
    info={'experiment':rec['experiment'], 'status':status, 'updated_utc':now_utc(), **extra}
    (rec['folder']/'run_status.json').write_text(json.dumps(info,indent=2,default=str))

STARTED=0
for index,rec in enumerate(TRIAL_PLAN,1):
    state, ev = outputs_state(rec['folder'])
    print(f'[{index}/{len(TRIAL_PLAN)}] {rec["experiment"]} -> {state}')
    if state in ('COMPLETE','COMPLETE_NO_VALIDATION'):
        if state == 'COMPLETE_NO_VALIDATION':
            print('  WARNING: CSVs found but no validation judge row. Comparison cannot select this trial.')
        continue
    if state == 'INCOMPLETE' and INCOMPLETE_ACTION == 'skip':
        print('  SKIPPED: incomplete outputs preserved. Inspect logs/checkpoints.')
        continue
    if DRY_RUN:
        print('  DRY RUN: would stage and train this trial; no writes or model/API calls.')
        continue
    if STARTED >= MAX_NEW_RUNS_PER_EXECUTION:
        print('  LIMIT reached; run this cell again to advance the queue.')
        break
    cfg_path=stage_trial(rec)
    if state == 'INCOMPLETE':
        old_outputs=rec['folder']/'outputs'
        archive=rec['folder']/('outputs_archive_'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ'))
        assert old_outputs.exists() and not archive.exists()
        old_outputs.rename(archive)
        print('  Archived incomplete outputs:',archive)
    write_status(rec,'RUNNING',command=f'{sys.executable} {CODE / "main.py"} --phase 1 --config {cfg_path} --device_mode colab')
    STARTED += 1
    cmd=[sys.executable,str(CODE/'main.py'),'--phase','1','--config',str(cfg_path), '--device_mode','colab']
    log_file=rec['folder']/'pipeline.log'
    print('  Starting training + evaluation. Log:',log_file)
    try:
        t0=time.monotonic()
        with open(log_file,'a',encoding='utf-8') as out:
            out.write('\n=== '+now_utc()+' ===\n')
            out.flush()
            proc=subprocess.Popen(cmd, cwd=str(CODE),env=os.environ.copy(),
                                  stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
                                  text=True,errors='replace',bufsize=1)
            assert proc.stdout is not None
            for line in proc.stdout:
                print(line,end='')
                out.write(line)
                out.flush()
            returncode=proc.wait()
        minutes=(time.monotonic()-t0)/60
    except BaseException as exc:
        write_status(rec,'INTERRUPTED',error=repr(exc))
        raise
    if returncode != 0:
        write_status(rec,'FAILED',returncode=returncode, wall_minutes=minutes)
        print('  FAILED:',returncode,'| saved outputs remain untouched')
        if STOP_ON_ERROR:
            raise RuntimeError(f'Failed: {rec["experiment"]}; inspect {log_file}')
        continue
    final_state, final_ev=outputs_state(rec['folder'])
    write_status(rec,final_state,returncode=returncode,wall_minutes=minutes,
                 evaluation_dir=str(final_ev) if final_ev else None)
    print('  Exit 0,',final_state, '|',round(minutes,1),'min')
    if final_state == 'INCOMPLETE':
        print('  WARNING: pipeline returned success but expected evaluation CSVs are missing.')
        if STOP_ON_ERROR:
            raise RuntimeError('Missing expected evaluation files; inspect pipeline.log')

print('New trials launched in this execution:',STARTED)

Streaming output truncated to the last 5000 lines.
Batches: 100%|██████████| 1/1 [00:00<00:00, 81.75it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 83.52it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 78.60it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 80.06it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 87.51it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 74.84it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 77.16it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 74.88it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 59.34it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 73.99it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 81.97it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 88.38it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 84.42it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 85.44it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 90.92it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 84.36it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 69.41it/s

## 6. Compare completed trials using **validation**, not test

The ranking is based on the **fine-tuned model's validation judge composite (out of 5)**. It also shows faithfulness, specificity, ROUGE-L and latency where available. Results are sorted by score, then faithfulness, then latency. A saved CSV only containing `test` cannot win the search.

This cell saves `search_summary.csv` and, when possible, `best_experiment.json` under `MyDrive/slm-distillation/experiments/hpo_smol_search/`. The winner is the **best observed validation trial**, not a guarantee of superior real-world performance. Avoid choosing on tiny score differences with ~37 clusters.

In [ ]:
from pathlib import Path
import pandas as pd, json

rows=[]
for rec in TRIAL_PLAN:
    state, ev = outputs_state(rec['folder'])
    t=rec['trial']
    row={
        'experiment':rec['experiment'],'model':t['model'],
        'learning_rate':t['learning_rate'],'epochs':t['epochs'],
        'lora_r':t['lora_r'],'lora_alpha':t['lora_alpha'],
        'state':state,'judge_val':float('nan'), 'faithfulness_val':float('nan'),
        'specificity_val':float('nan'),'rouge_l_val':float('nan'),
        'latency_sec':float('nan'), 'train_minutes':float('nan'),
        'evaluation_dir':str(ev) if ev else ''
    }
    if ev and state == 'COMPLETE':
        try:
            judge=pd.read_csv(ev/'judge_summary.csv')
            judge['split_clean']=judge['split'].astype(str).str.lower().str.strip()
            judge['model_clean']=judge['model'].astype(str).str.lower().str.strip()
            j=judge[judge['split_clean'].isin(VAL_ALIASES) & (judge['model_clean']=='finetuned')]
            assert len(j)==1, f'Expected 1 validation finetuned judge row, got {len(j)}'
            for dest,src in [('judge_val','composite'),('faithfulness_val','faithfulness'),
                             ('specificity_val','specificity')]:
                if src in j.columns: row[dest]=float(j.iloc[0][src])
            metrics=pd.read_csv(ev/'metrics_summary.csv')
            if {'split','model','rouge_l_same'}.issubset(metrics.columns):
                m=metrics[metrics['split'].astype(str).str.lower().str.strip().isin(VAL_ALIASES) &
                          (metrics['model'].astype(str).str.lower().str.strip()=='finetuned')]
                if len(m)==1: row['rouge_l_val']=float(m.iloc[0]['rouge_l_same'])
            biz=pd.read_csv(ev/'business_eval.csv')
            if {'metric','value'}.issubset(biz.columns):
                b=dict(zip(biz['metric'],pd.to_numeric(biz['value'],errors='coerce')))
                row['latency_sec']=b.get('slm_inference_latency_mean_s_finetuned',float('nan'))
                row['train_minutes']=b.get('finetuning_wall_time_min',float('nan'))
        except Exception as exc:
            row['state']='EVALUATION_PARSE_ERROR'
            row['error']=str(exc)
    rows.append(row)

summary=pd.DataFrame(rows)
valid=summary[(summary['state']=='COMPLETE') & pd.notna(summary['judge_val'])].copy()
valid=valid.sort_values(['judge_val','faithfulness_val','latency_sec'],
                        ascending=[False,False,True],na_position='last')

print('All planned experiments:')
display(summary.drop(columns=['evaluation_dir']).round(4))
print('\nRankable experiments (validation judge):')
if valid.empty:
    print('None yet. If CSVs exist but are COMPLETE_NO_VALIDATION, your pipeline must emit validation scores.')
else:
    display(valid[['experiment','model','learning_rate','epochs','lora_r','lora_alpha',
                   'judge_val','faithfulness_val','specificity_val','rouge_l_val',
                   'latency_sec','train_minutes']].round(4))

SEARCH_ROOT.mkdir(parents=True,exist_ok=True)
summary.to_csv(SEARCH_ROOT/'search_summary.csv',index=False)
print('Saved:',SEARCH_ROOT/'search_summary.csv')
if not valid.empty:
    top=valid.iloc[0]
    winner={
        'selection':'highest observed validation judge composite; tie-break faithfulness then latency',
        'experiment':top['experiment'], 'model':top['model'],
        'judge_val':float(top['judge_val']),
        'config':str(EXPERIMENTS/top['experiment']/'experiment_config.yaml'),
        'evaluation_dir':top['evaluation_dir'],
        'ranked_count':len(valid),
        'selected_utc':now_utc(),
        'caveat':'validation-only selection, not a definitive held-out test result',
    }
    (SEARCH_ROOT/'best_experiment.json').write_text(json.dumps(winner,indent=2))
    print('Best observed validation trial:',top['experiment'],'score:',top['judge_val'])
    print('Saved:',SEARCH_ROOT/'best_experiment.json')
else:
    old=SEARCH_ROOT/'best_experiment.json'
    if old.exists():
        print('WARNING: previous best_experiment.json exists; it may be stale for this plan.')

All planned experiments:


,experiment,model,learning_rate,epochs,lora_r,lora_alpha,state,judge_val,faithfulness_val,specificity_val,rouge_l_val,latency_sec,train_minutes
0,hpo_360m_base_a54f178d4227,360m,0.0002,3,16,16,COMPLETE,2.4583,3.25,2.125,0.1827,0.4474,3.8480
1,hpo_360m_lr_1e4_57c233a7b33d,360m,0.0001,3,16,16,COMPLETE,2.3333,3.25,2.000,0.1524,0.4893,3.8912
2,hpo_17b_base_c66db95a7295,17b,0.0002,3,16,16,COMPLETE,2.6250,3.75,2.125,0.3479,0.2820,2.9792
3,hpo_17b_lr_1e4_e503eba19d34,17b,0.0001,3,16,16,COMPLETE,2.7500,3.75,2.375,0.3557,0.2693,2.9222



Rankable experiments (validation judge):


,experiment,model,learning_rate,epochs,lora_r,lora_alpha,judge_val,faithfulness_val,specificity_val,rouge_l_val,latency_sec,train_minutes
3,hpo_17b_lr_1e4_e503eba19d34,17b,0.0001,3,16,16,2.7500,3.75,2.375,0.3557,0.2693,2.9222
2,hpo_17b_base_c66db95a7295,17b,0.0002,3,16,16,2.6250,3.75,2.125,0.3479,0.2820,2.9792
0,hpo_360m_base_a54f178d4227,360m,0.0002,3,16,16,2.4583,3.25,2.125,0.1827,0.4474,3.8480
1,hpo_360m_lr_1e4_57c233a7b33d,360m,0.0001,3,16,16,2.3333,3.25,2.000,0.1524,0.4893,3.8912


Saved: /content/drive/MyDrive/slm-distillation/experiments/hpo_smol_search/search_summary.csv
Best observed validation trial: hpo_17b_lr_1e4_e503eba19d34 score: 2.75
Saved: /content/drive/MyDrive/slm-distillation/experiments/hpo_smol_search/best_experiment.json


## 7. Inspect the winner and its held-out test results **after selection**

Do **not** use test scores to adjust trials after viewing this. The existing pipeline may evaluate the test split for every run; this cell only *displays* the winner's saved test results after validation selection. If the files don't contain a test split, it says so explicitly.

In [ ]:
if valid.empty:
    print('No validation-ranked winner; no test results shown.')
else:
    best=valid.iloc[0]
    ev=Path(best['evaluation_dir'])
    print('Selected from validation only:',best['experiment'])
    print('Config:',EXPERIMENTS/best['experiment']/'experiment_config.yaml')
    for filename in ['judge_summary.csv','metrics_summary.csv']:
        df=pd.read_csv(ev/filename)
        if 'split' not in df:
            print(filename,': no split column')
            continue
        test=df[df['split'].astype(str).str.lower().str.strip().eq('test')]
        print('\n',filename,'(test only):')
        if test.empty:
            print('No test rows saved.')
        else:
            display(test)

Selected from validation only: hpo_17b_lr_1e4_e503eba19d34
Config: /content/drive/MyDrive/slm-distillation/experiments/hpo_17b_lr_1e4_e503eba19d34/experiment_config.yaml

 judge_summary.csv (test only):


,split,model,equivalence,faithfulness,specificity,composite
6,test,teacher,5.000000,5.000000,5.000000,5.000000
7,test,baseline,2.416667,4.083333,2.416667,2.972222
8,test,finetuned,2.250000,3.583333,2.083333,2.638889



 metrics_summary.csv (test only):


,split,model,cosine_sim_same,cosine_sim_multi,rouge_l_same,rouge_l_multi,bertscore_f1_same,bertscore_f1_multi
6,test,teacher,1.000000,1.000000,1.000000,1.000000,NaN,NaN
7,test,baseline,0.606962,0.649517,0.370912,0.446623,NaN,NaN
8,test,finetuned,0.669148,0.706022,0.414491,0.500624,NaN,NaN


## Practical notes / limitations

- **New parameter choices**: Edit `CUSTOM_TRIALS`, select `'custom'`, and run plan → execution → comparison again. Your experiment IDs are content-derived, so prior trials remain untouched. Changing the same parameter values or source inputs creates no duplicate trial.
- **Resume after Colab disconnect**: Rerun the setup, controls, dataset, plan and execute cells. Completed output folders will be skipped. Incomplete jobs are preserved until you explicitly choose `archive_and_restart`. **This is restart, not checkpoint resume.**
- **Selection and ranking**: Requires `judge_summary.csv` with one row where `split` is `validation`/`val`/`valid`/`dev` and `model` is `finetuned`. Exact split names and evaluation content depend on your GitHub checkout. If no validation rows are emitted, inspect the repository rather than ranking on test scores.
- **Performance/cost**: No exact runtime or API cost can be predicted without executing your pipeline. Default 4 (quick) / 8 (extended) full pipelines can be expensive. Test a single trial first (`MAX_NEW_RUNS_PER_EXECUTION=1`). The `business_eval.csv` latency is an average from the pipeline, not a repeated independent latency benchmark.
- **Statistical caution**: The original 1,000-ticket experiment has ~37 clusters. A 15% validation split contains few independent clusters; changes of a few hundredths in composite are not a reliable win. More held-out clusters / replicate runs would be needed for stronger evidence.
- **Security**: This notebook uses the existing project secrets (Anthropic/HF), but does not print them. The repository is cloned through a clean public URL, not a Git remote embedding your GitHub PAT.
- **No changes to your GitHub repo are committed or pushed**. Compatibility patches apply only to the Colab checkout, as in your earlier notebook. No results were actually trained by creating this notebook.